In [1]:
import torch
import torch.nn as nn
from torch.distributed._shard.sharding_spec.chunk_sharding_spec_ops import embedding

## 构造参数

In [2]:
rnn = nn.RNN(
    input_size=3,
    hidden_size=4,
    num_layers=1,
    bidirectional=True,
    batch_first=True
)

## 前向传播

In [10]:
# 定义输入层
vocab_size = 2000
N = 2
L = 4
input_ids = torch.randint(vocab_size, (N, L))
print(input_ids)

tensor([[1974, 1992, 1254,  710],
        [ 141, 1491,  605, 1188]])


In [13]:
# 词嵌入层
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=3)

In [15]:
# 嵌入层向前传播,得到词向量的列表
input = embedding(input_ids)
print(input)
print(input.shape)

tensor([[[-0.2643,  0.4392, -1.6028],
         [ 0.6073,  1.9533,  0.7379],
         [ 0.1992, -1.0014, -0.0065],
         [-0.3948,  1.0120, -0.1736]],

        [[ 1.0008, -0.4085,  0.2722],
         [-1.5049, -0.5182,  0.2917],
         [ 0.0708, -2.1455, -0.2772],
         [-0.0252, -1.1242, -0.2373]]], grad_fn=<EmbeddingBackward0>)
torch.Size([2, 4, 3])


## 1. 单层单向

In [16]:
rnn1 = nn.RNN(3,4,batch_first=True)

In [17]:
out_put,hn = rnn1(input)

In [18]:
print(out_put.shape)
print(hn.shape)

torch.Size([2, 4, 4])
torch.Size([1, 2, 4])


In [19]:
print(out_put)

tensor([[[-0.5387,  0.5925, -0.4079, -0.7958],
         [-0.4543,  0.9434, -0.2952, -0.3276],
         [-0.3958,  0.4107, -0.8399,  0.0786],
         [-0.5735,  0.8077, -0.1644, -0.3245]],

        [[ 0.0922,  0.0545, -0.6879, -0.2768],
         [-0.6485,  0.6754, -0.3394,  0.4227],
         [-0.0988, -0.3273, -0.8213,  0.0523],
         [ 0.0525, -0.1319, -0.6750,  0.0578]]], grad_fn=<TransposeBackward1>)


In [20]:
print(hn)

tensor([[[-0.5735,  0.8077, -0.1644, -0.3245],
         [ 0.0525, -0.1319, -0.6750,  0.0578]]], grad_fn=<StackBackward0>)


## 2. 两层单向

In [21]:
rnn2 = nn.RNN(3,4,batch_first=True,num_layers=2)

In [23]:
out_put,hn = rnn2(input)
print(out_put.shape)
print(hn.shape)

torch.Size([2, 4, 4])
torch.Size([2, 2, 4])


In [24]:
print(out_put)

tensor([[[ 0.2235,  0.0527, -0.4634,  0.3978],
         [ 0.5965, -0.3620, -0.0687, -0.0630],
         [ 0.2743,  0.0208, -0.5641,  0.1141],
         [ 0.6558, -0.5536, -0.1773, -0.2027]],

        [[ 0.3804, -0.1037, -0.3519,  0.2069],
         [ 0.5473, -0.1739, -0.1959, -0.4596],
         [ 0.5780, -0.4843, -0.5664, -0.0503],
         [ 0.5059, -0.4582, -0.5509, -0.0730]]], grad_fn=<TransposeBackward1>)


In [25]:
print(hn)

tensor([[[ 0.4040, -0.2681, -0.0078,  0.3104],
         [-0.3368,  0.1826,  0.6448,  0.4333]],

        [[ 0.6558, -0.5536, -0.1773, -0.2027],
         [ 0.5059, -0.4582, -0.5509, -0.0730]]], grad_fn=<StackBackward0>)


## 3. 多层双向

In [26]:
rnn2 =nn.RNN(3,4,batch_first=True,num_layers=2,bidirectional=True)
out_put,hn = rnn2(input)

In [28]:
print(out_put.shape)
print(hn.shape)

torch.Size([2, 4, 8])
torch.Size([4, 2, 4])


In [29]:
print(out_put)

tensor([[[ 0.3835,  0.1207,  0.8926,  0.4601, -0.8800, -0.5924,  0.3715,
          -0.2738],
         [-0.2783,  0.8354,  0.8035, -0.1061, -0.8166, -0.2239,  0.8345,
          -0.9062],
         [ 0.7526,  0.6754,  0.3887, -0.0199, -0.5233, -0.4768,  0.3389,
           0.3023],
         [-0.4068,  0.7661,  0.8302,  0.1049, -0.8202, -0.1681,  0.8152,
          -0.9114]],

        [[ 0.4764,  0.2046,  0.7453, -0.4149, -0.6055, -0.0100,  0.3831,
          -0.4662],
         [ 0.3816,  0.4747,  0.3429,  0.5377, -0.0992,  0.3288,  0.6026,
          -0.6766],
         [ 0.5377,  0.3641,  0.0781,  0.2155, -0.1043, -0.2617,  0.2302,
          -0.1623],
         [ 0.2114,  0.4224,  0.6219,  0.0321, -0.2776,  0.3056,  0.3521,
          -0.7628]]], grad_fn=<TransposeBackward1>)


In [30]:
print(hn)

tensor([[[-6.2910e-01,  6.3275e-01, -3.5031e-01, -7.2120e-01],
         [-3.3990e-01,  7.9119e-04, -3.2685e-01,  8.4577e-02]],

        [[-3.4537e-01, -8.5674e-01, -6.7887e-01,  5.7725e-01],
         [ 2.8902e-01, -4.1678e-01,  3.3909e-01,  3.2925e-01]],

        [[-4.0680e-01,  7.6612e-01,  8.3016e-01,  1.0495e-01],
         [ 2.1140e-01,  4.2243e-01,  6.2194e-01,  3.2081e-02]],

        [[-8.8000e-01, -5.9237e-01,  3.7148e-01, -2.7385e-01],
         [-6.0550e-01, -1.0025e-02,  3.8312e-01, -4.6620e-01]]],
       grad_fn=<StackBackward0>)
